In [ ]:
#The goal of this script is to identify clusters in the average contaminant concentrations in springs.
#Before this script, the data was cleanedup and processed in R.

#Import libraries ----
import os
os.environ["OMP_NUM_THREADS"] = "1"
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import math
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
sns.set_theme()

#Read in Data ----
springData = pd.read_csv("SelectContaminantAveragesfromSprings.csv") #Data has already been scaled based on z-score

#Finish preparing data ----
springIDs = springData.iloc[:,0]
print(springIDs)
springData_noIDs = springData.iloc[:,1:]

#Choose number of clusters ----
wcss = [] #Within Cluster Sum of Squares
silhouette = [] #Average Silhouette Score
calinski = [] #Calinski Harabasz Score
k_range = range(2,8) #Range of cluster numbers to iterate over

for k in k_range:
    km = KMeans(n_clusters = k, n_init = 25, random_state = 1234)
    km.fit(springData_noIDs)
    wcss.append(km.inertia_)
    silhouette.append(silhouette_score(springData_noIDs,km.labels_))
    calinski.append(calinski_harabasz_score(springData_noIDs, km.labels_))

wcss_series = pd.Series(wcss, index = k_range)
silhouette_series = pd.Series(silhouette,index = k_range)
calinski_series = pd.Series(calinski, index = k_range)

def identifyClustersViaPlot(series,ylab):
    """This function plots the results of a test over various cluster numbers.

    Parameters
    ----------
    series: pandas Series
        A series of the scores across different k values
    ylab: str
        The prefered label of the y-axis (what test was done)
    Returns
    -------
    None
        Creates a plot

    Example
    -------
    >>> identifyClustersViaPlot(series = wcss_series,ylab = "Within Cluster Sum of Squares (WCSS)")
    """
    plt.figure(figsize=(8,6))
    ax = sns.lineplot(y = series, x = series.index)
    ax = sns.scatterplot(y = series, x = series.index, s = 150)
    ax = ax.set(xlabel = "Number of Clusters (k)",
                ylabel = ylab)
    
identifyClustersViaPlot(series = wcss_series,ylab = "Within Cluster Sum of Squares (WCSS)")
#WCSS suggests 4 clusters is appropriate, but not very clear (most significant inflection point)
identifyClustersViaPlot(series = silhouette_series, ylab = "Average Silhoette Score")
#Silhoette Score suggests 2 clusters (Highest value)
identifyClustersViaPlot(series = calinski_series, ylab = "Calinski Harabasz Score")
#Calinksi Harabasz Score suggests 4 clusters (first abrupt elbow)
n_springs = len(springData)
k = int(round(math.sqrt(n_springs/2),0)) #Basic estimation 
print(k) #Basic estimation suggests 2 clusters

#Creating 4 clusters ----
km = KMeans(n_clusters = 4, n_init = 25, random_state = 1234)
km.fit(springData_noIDs)
#km.cluster_centers_
springData["Cluster"] = km.labels_.tolist()

#Export cluster identificiation to csv ----
springData.to_csv('SelectContaminantAveragesfromSprings_withCluster.csv')